<a href="https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


The five contract answers, in plain words:

1. One row = one page on one day (day x client x page). For the model I add the days up, so one row = one page in March 2026.
2. Table: fact_content_daily_performance, only the month=2026-03 part.
3. Time window: feature window 2026-03-01 to 2026-03-21. Outcome window 2026-03-22 to 2026-03-31. The two never overlap.
4. What I rank (label): a page "declined" if its daily impressions in the outcome window are below 80% of its daily impressions in the feature window. This is a future outcome I measure, not a rule on today's numbers. The 80% cut and the 100-impression floor are my own choices.
5. Deliberately excluded: June 2026, the final month, and the _sample table. They stay sealed for a final test.

In [1]:
%pip -q install duckdb
import os
import duckdb

# The token comes from the Colab Secret named HF_TOKEN (key icon on the left). Never paste it here.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Add a Colab Secret named HF_TOKEN (key icon on the left) and turn on notebook access."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("Connected. Slice = daily performance, month 2026-03.")

Connected. Slice = daily performance, month 2026-03.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Every field I touch goes in one bucket. The table below shows them.
- Features come only from the feature window (Mar 1-21).
- The label comes only from the outcome window (Mar 22-31).
- IDs are for grouping only, never for learning.
- Excluded fields each have a reason.

In [2]:
import pandas as pd

fields = pd.DataFrame([
    ("gsc_impressions", "feature", "added up over Mar 1-21 only"),
    ("gsc_clicks", "feature", "used only to make ctr, Mar 1-21 only"),
    ("gsc_avg_position", "feature", "averaged over Mar 1-21 only"),
    ("gsc_impressions (Mar 22-31)", "label", "the outcome window: did views drop?"),
    ("report_date", "context", "cuts the two windows"),
    ("client_hash_id", "context", "groups pages by client for a fair split, never a feature"),
    ("content_hash_id", "context", "page id, never a feature"),
    ("ga4_data_available and GA4 columns", "excluded", "GA4 only exists for part of the history, and the flag has 3 values"),
    ("June 2026 and the _sample table", "excluded", "sealed for a final test, not for building"),
], columns=["field", "bucket", "why"])
print(fields.to_string(index=False))

                             field   bucket                                                                why
                   gsc_impressions  feature                                        added up over Mar 1-21 only
                        gsc_clicks  feature                               used only to make ctr, Mar 1-21 only
                  gsc_avg_position  feature                                        averaged over Mar 1-21 only
       gsc_impressions (Mar 22-31)    label                                the outcome window: did views drop?
                       report_date  context                                               cuts the two windows
                    client_hash_id  context           groups pages by client for a fair split, never a feature
                   content_hash_id  context                                           page id, never a feature
ga4_data_available and GA4 columns excluded GA4 only exists for part of the history, and the flag has 3 values
 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


Three queries, all on March 2026: (1) the grain, (2) row count and date span, (3) availability using IS TRUE (the GA4 flag can be TRUE, FALSE, or empty, so = FALSE would miss the empty ones).

Five features (all from Mar 1-21 only), and why each is known at the decision moment (the end of Mar 21):
1. imp_21d: total impressions in the feature window. Known because it only adds up days up to Mar 21.
2. ctr_21d: clicks divided by impressions in the feature window. Same reason.
3. pos_21d: average position in the feature window. Same reason.
4. active_days_21d: days with at least one impression, up to Mar 21.
5. momentum_7v14: impressions per day in Mar 15-21 divided by impressions per day in Mar 1-14. Both parts end on Mar 21.

The trap: I add one column made from the label (outcome window vs feature window). The score jumps to almost perfect, which is leakage, because the column contains the answer. Then I delete it and keep the honest number.

In [3]:
# Query 1 - grain: is one row really one page on one day?
import requests
h = {"Authorization": f"Bearer {HF_TOKEN}"}
print("Token belongs to:", requests.get("https://huggingface.co/api/whoami-v2", headers=h).json().get("name"))
url = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_clients.parquet"
print("File check:", requests.head(url, headers=h, allow_redirects=True).status_code)
dups = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Query 1 - duplicate (day, client, page) rows found:", len(dups), "(0 means the grain holds)")

# Query 2 - size and date span
q2 = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_pages
    FROM {MARCH}
""").df()
print("\nQuery 2 - size and dates")
print(q2.to_string(index=False))

# Query 3 - availability: the GA4 flag has three values, so use IS TRUE
q3 = con.sql(f"""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)         AS rows_with_impressions
    FROM {MARCH}
""").df()
print("\nQuery 3 - availability (IS TRUE keeps only rows where GA4 data is really there)")
print(q3.to_string(index=False))

Token belongs to: jayrasal28
File check: 200


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 - duplicate (day, client, page) rows found: 0 (0 means the grain holds)

Query 2 - size and dates
 n_rows  first_day   last_day  n_clients  n_pages
9841378 2026-03-01 2026-03-31         55   331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3 - availability (IS TRUE keeps only rows where GA4 data is really there)
 all_rows  ga4_true  ga4_false  ga4_null  rows_with_impressions
  9841378    413966    6408671   3018741                3611061


In [4]:
# One row = one page. Features use Mar 1-21 only. The outcome uses Mar 22-31 only.
data = con.sql(f"""
    WITH pages AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-21' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_21d,
            SUM(CASE WHEN report_date <= DATE '2026-03-21' THEN COALESCE(gsc_clicks, 0) ELSE 0 END)      AS clk_21d,
            AVG(CASE WHEN report_date <= DATE '2026-03-21' THEN gsc_avg_position END)                    AS pos_21d,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-21' AND COALESCE(gsc_impressions, 0) > 0
                                THEN report_date END)                                                     AS active_days_21d,
            SUM(CASE WHEN report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-21' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last7,
            SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_first14,
            SUM(CASE WHEN report_date >= DATE '2026-03-22' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_out_10d
        FROM {MARCH}
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, imp_21d, imp_out_10d, pos_21d, active_days_21d,
           clk_21d / NULLIF(imp_21d, 0)                          AS ctr_21d,
           (imp_last7 / 7.0) / NULLIF(imp_first14 / 14.0, 0)     AS momentum_7v14
    FROM pages
    WHERE imp_21d >= 100
""").df()

# Label: views per day in the outcome window fell below 80% of the feature window.
data["declined"] = ((data.imp_out_10d / 10) < 0.8 * (data.imp_21d / 21)).astype(int)

FEATURES = ["imp_21d", "ctr_21d", "pos_21d", "active_days_21d", "momentum_7v14"]
data = data.dropna(subset=FEATURES).reset_index(drop=True)
print(f"{len(data):,} pages with 100+ impressions in Mar 1-21")
print(f"Share that declined in Mar 22-31: {data.declined.mean():.1%}")
print(data[FEATURES + ["declined"]].head(5).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

86,927 pages with 100+ impressions in Mar 1-21
Share that declined in Mar 22-31: 31.5%
 imp_21d  ctr_21d  pos_21d  active_days_21d  momentum_7v14  declined
   309.0 0.006472 4.553920               19       1.452514         0
   611.0 0.001637 4.348539               19       0.758465         1
  1188.0 0.003367 1.948582               20       1.346479         0
   359.0 0.002786 4.388223               19       1.263636         0
   321.0 0.009346 3.414565               19       1.086538         0


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
                       .split(data, data.declined, groups=data.client_hash_id))
    tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(data.loc[train, cols], data.declined[train])
    return roc_auc_score(data.declined[test], tree.predict_proba(data.loc[test, cols])[:, 1])

# The trap: ONE column made from the label (outcome window vs feature window).
data["outcome_vs_feature"] = (data.imp_out_10d / 10) / (data.imp_21d / 21)
print(f"With the leaky column:  AUC = {quick_score(FEATURES + ['outcome_vs_feature']):.3f}  <- looks perfect, but it contains the answer")

# Delete it and keep the honest number.
data = data.drop(columns=["outcome_vs_feature"])
print(f"Honest (5 features):    AUC = {quick_score(FEATURES):.3f}  <- the number I keep")
print("(AUC: 0.5 = guessing, 1.0 = perfect. Test clients were never seen in training.)")

With the leaky column:  AUC = 1.000  <- looks perfect, but it contains the answer
Honest (5 features):    AUC = 0.780  <- the number I keep
(AUC: 0.5 = guessing, 1.0 = perfect. Test clients were never seen in training.)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


Named limitation: this slice is one month (March 2026) with a 10-day outcome window. That is short and noisy, and one month cannot show seasonality, so a page that "declined" may just have had a normal dip.

Also, clients start tracking on different dates (unbalanced history), so some pages may be missing early days.

I can say what I observed in this slice (directional). I cannot say the results hold for other months.

In [6]:
print("Feature window: 2026-03-01 to 2026-03-21 (21 days)")
print("Outcome window: 2026-03-22 to 2026-03-31 (10 days)")
print(f"Pages in this slice: {len(data):,} from one month only")

Feature window: 2026-03-01 to 2026-03-21 (21 days)
Outcome window: 2026-03-22 to 2026-03-31 (10 days)
Pages in this slice: 86,927 from one month only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.